In [ ]:
%%capture
!pip install -q kokoro>=0.9.4 soundfile

In [ ]:
%%capture
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

In [ ]:
from kokoro import KPipeline
from datasets import load_dataset
import soundfile as sf
import pandas as pd
import os
import numpy as np
import io
from tqdm import tqdm
import random


In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
dataset = load_dataset("username/wiki-ai-filtered")

dataset

In [ ]:
pipeline = KPipeline(lang_code='a')

def convert_text_to_speech(text, voice, output_file_path):
    generator = pipeline(
            text, 
            voice=voice, 
            speed=1, 
            split_pattern=r'\n+'
        )
    all_audio = []
    for _, _, audio in generator:
        all_audio.append(audio)
    
    if not all_audio:
        raise Exception(status_code=400, detail="No audio generated")

    final_audio = np.concatenate(all_audio)
    buffer = io.BytesIO()
    sf.write(buffer, final_audio, 24000, format='WAV')
    buffer.seek(0) # Reset pointer to start of file

    with open(output_file_path, 'wb') as f:
        f.write(buffer.read())
    
    return output_file_path



In [ ]:
os.makedirs("wiki-audio/train", exist_ok=True)

In [ ]:
VOICE_LIST = [
"af_heart",
"af_alloy",
"af_aoede",
"af_bella",
"af_jessica",
"af_kore",
"af_nicole",
"af_nova",
"af_river",
"af_sarah",
"af_sky",
"am_adam",
"am_echo",
"am_eric",
"am_fenrir",
"am_liam",
"am_michael",
"am_onyx",
"am_puck",
"am_santa",
"bf_alice",
"bf_emma",
"bf_isabella",
"bf_lily",
"bm_daniel",
"bm_fable",
"bm_george",
"bm_lewis"
]

In [ ]:
! ls wiki-audio/train/

In [ ]:
output_file_name_list = []
for row in tqdm(dataset['train'].select(range(100))):
    text = row['text']
    unique_id = row['id']
    output_file_path = f"wiki-audio/train/{unique_id}.wav"
    try:
        convert_text_to_speech(text, voice=random.choice(VOICE_LIST), output_file_path=output_file_path)
        output_file_name_list.append(f"{unique_id}.wav")
    except Exception as e:
        print(f"Error processing {unique_id}: {e}")
        continue

In [ ]:
! ls wiki-audio/train | wc -l

In [ ]:
df = dataset["train"].to_pandas()
temp_df = df[:100]
temp_df = temp_df[['text']]
temp_df = pd.concat([temp_df, pd.DataFrame({'file_name': output_file_name_list})], axis=1)
temp_df = temp_df[['file_name', 'text']]
temp_df.to_csv("wiki-audio/train/metadata.csv", index=False)

In [ ]:
audio_dataset = load_dataset("wiki-audio/train")
audio_dataset

In [ ]:
audio_dataset.push_to_hub("username/aiaudio")